# Kaggle T4 ×2 training

Before running this notebook, open **Kaggle → Notebook options → Accelerator** and select **GPU T4 ×2**. The next cell stops immediately if Kaggle has not allocated two CUDA devices. Training uses `torch.nn.DataParallel` across `cuda:0` and `cuda:1`; `batch_size` is the total batch size, split evenly across the two GPUs.

In [1]:
# Kaggle includes most packages. Uncomment only if this environment reports a missing package.
# %pip install -q numpy pandas scipy soundfile torchaudio safetensors scikit-learn

In [8]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchaudio
import soundfile as sf
from scipy.signal import resample_poly
from pathlib import Path
from copy import deepcopy
from safetensors.torch import save_file
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset


In [ ]:
# ===== Standalone project configuration =====
# Attach the project data in Kaggle, then set this to its directory under /kaggle/input.
# Expected layout: <DATA_ROOT>/crop_data/music_crop6.csv and audio paths below crop_data/.
DATA_ROOT = Path("/kaggle/input/ai-music-detection")
TRAIN_CSV = DATA_ROOT / "crop_data/music_crop6.csv"
TEST_CSV = DATA_ROOT / "crop_data/test.csv"
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inlined from config/model_hparams.yaml. Edit this cell instead of relying on YAML files.
DEFAULT_VARIANT = "alpha"
TRAINING_VARIANTS = [
    {"name": "alpha", "model_kwargs": {"input_spec_dim": 128, "input_temp_dim": 188, "embed_dim": 768, "f_clip": 1, "t_clip": 3, "num_heads": 6, "num_layers": 12, "dim_feedforward": 3072, "pos_drop_rate": 0.2}},
    {"name": "beta", "model_kwargs": {"input_spec_dim": 128, "input_temp_dim": 188, "embed_dim": 768, "f_clip": 3, "t_clip": 5, "num_heads": 6, "num_layers": 12, "dim_feedforward": 3072, "pos_drop_rate": 0.0}},
    {"name": "gamma", "model_kwargs": {"input_spec_dim": 128, "input_temp_dim": 188, "embed_dim": 768, "f_clip": 5, "t_clip": 7, "num_heads": 6, "num_layers": 12, "dim_feedforward": 3072, "pos_drop_rate": 0.0}},
]

def get_training_variants():
    return deepcopy(TRAINING_VARIANTS)

def get_training_default_variant():
    return DEFAULT_VARIANT

def load_manifest(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}. Set DATA_ROOT in the configuration cell.")
    df = pd.read_csv(csv_path)
    required_columns = {"filename", "path", "fake_type", "label"}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")
    df = df.copy()
    df["path"] = df["path"].map(
        lambda value: str(Path(value)) if Path(value).is_absolute() else str(DATA_ROOT / value)
    )
    return df

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Training CSV: {TRAIN_CSV}")

In [ ]:
# ===== Standalone audio Dataset =====
class SonicDataset(Dataset):
    def __init__(self, df, sample_rate=16000, duration_seconds=6.0):
        self.df = df.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.expected_samples = int(sample_rate * duration_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename, audio_path = row["filename"], row["path"]
        fake_type, label = row["fake_type"], int(row["label"])
        try:
            wave, sample_rate = sf.read(audio_path, dtype="float32", always_2d=True)
        except Exception as exc:
            raise RuntimeError(f"Failed to load audio '{audio_path}': {exc}") from exc

        waveform = torch.from_numpy(wave.T)
        waveform = waveform.mean(dim=0) if waveform.size(0) > 1 else waveform.squeeze(0)
        if int(sample_rate) != self.sample_rate:
            waveform = torch.from_numpy(np.asarray(
                resample_poly(waveform.numpy(), up=self.sample_rate, down=int(sample_rate)),
                dtype=np.float32,
            ))

        waveform = waveform.to(torch.float32)
        if waveform.numel() < self.expected_samples:
            waveform = nn.functional.pad(waveform, (0, self.expected_samples - waveform.numel()))
        else:
            waveform = waveform[:self.expected_samples]

        return {
            "x": waveform, "y": torch.tensor(label, dtype=torch.long),
            "filename": filename, "fake_type": fake_type, "path": audio_path,
        }

In [ ]:
# ===== Standalone model, tokenizer and positional encoding =====
class SinusoidPosEncoding(nn.Module):
    def __init__(self, token_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, token_dim)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, token_dim, 2, dtype=torch.float32) * (-torch.log(torch.tensor(10000.0)) / token_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Tokenizer1D(nn.Module):
    def __init__(self, input_dim, embed_dim, clip_size):
        super().__init__()
        self.conv1d = nn.Conv1d(input_dim, embed_dim, kernel_size=clip_size, stride=clip_size)
        self.act = nn.GELU()
        self.pos_encoder = SinusoidPosEncoding(embed_dim)

    def forward(self, x):
        return self.pos_encoder(self.act(self.conv1d(x)).transpose(1, 2))

class STTokenizer(nn.Module):
    def __init__(self, input_spec_dim, input_temp_dim, t_clip, f_clip, embed_dim):
        super().__init__()
        self.temporal_tokenizer = Tokenizer1D(input_spec_dim, embed_dim, t_clip)
        self.spectro_tokenizer = Tokenizer1D(input_temp_dim, embed_dim, f_clip)

    def forward(self, x):
        temporal = self.temporal_tokenizer(x)
        spectral = self.spectro_tokenizer(x.permute(0, 2, 1))
        return torch.cat((temporal, spectral), dim=1)

class SpecTTTra(nn.Module):
    def __init__(self, input_spec_dim, input_temp_dim, embed_dim, f_clip, t_clip, num_heads, num_layers, dim_feedforward=2048, num_classes=2, sample_rate=16000, n_fft=2048, hop_length=512, f_min=20.0, f_max=8000.0, expected_samples=96000, pos_drop_rate=0.0):
        super().__init__()
        self.expected_samples, self.input_temp_dim = expected_samples, input_temp_dim
        self.mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=input_spec_dim, f_min=f_min, f_max=f_max, center=True, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.tokenizer = STTokenizer(input_spec_dim, input_temp_dim, t_clip, f_clip, embed_dim)
        self.pos_drop = nn.Dropout(pos_drop_rate)
        self.transformer_layers = nn.TransformerEncoderLayer(embed_dim, num_heads, dim_feedforward=dim_feedforward, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layers, num_layers=num_layers)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        if x.ndim == 3 and x.size(1) == 1:
            x = x.squeeze(1)
        if x.ndim != 2 or x.size(1) != self.expected_samples:
            raise ValueError(f"Expected waveform [B, {self.expected_samples}], got {tuple(x.shape)}")
        mel = self.to_db(self.mel_transform(x.to(torch.float32)))
        if mel.size(-1) < self.input_temp_dim:
            mel = nn.functional.pad(mel, (0, self.input_temp_dim - mel.size(-1)))
        else:
            mel = mel[:, :, :self.input_temp_dim]
        tokens = self.pos_drop(self.tokenizer(mel))
        encoded = self.transformer(tokens).transpose(1, 2)
        return self.classifier(self.pooling(encoded).squeeze(-1))

In [9]:
# Kaggle must be configured with Accelerator = GPU T4 x2 before this cell is run.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle, select Notebook options → Accelerator → GPU T4 x2.")

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(
        f"Expected two GPUs, but Kaggle allocated {GPU_COUNT}. "
        "Select Notebook options → Accelerator → GPU T4 x2, then restart the session."
    )

DEVICE_IDS = [0, 1]
device = torch.device("cuda:0")
torch.cuda.set_device(device)
torch.backends.cudnn.benchmark = True  # Audio clips have a fixed 6-second shape.
torch.set_float32_matmul_precision("high")

print(f"Using {len(DEVICE_IDS)} GPUs with DataParallel: " + ", ".join(
    f"cuda:{i} ({torch.cuda.get_device_name(i)})" for i in DEVICE_IDS
))

cuda


In [10]:
variants = get_training_variants("clip_6s_variants")
default_variant = get_training_default_variant("clip_6s_default_variant")

hyper_params = [v["model_kwargs"] for v in variants]
variant_names = [v.get("name", f"v{i+1}") for i, v in enumerate(variants)]

print("Available variants:", variant_names)
print("Default variant:", default_variant)

Available variants: ['alpha', 'beta', 'gamma']
Default variant: alpha


In [11]:
df = load_manifest(TRAIN_CSV)
data = SonicDataset(df, duration_seconds=6.0)

train_loader = DataLoader(data, batch_size=32, shuffle=True, num_workers=2)
batch = next(iter(train_loader))
X = batch["x"]
y = batch["y"]
print(X.shape, y.shape)

num_zeros = (df["label"] == 0).sum().item()
num_ones = (df["label"] == 1).sum().item()

print("Số label 0:", num_zeros)
print("Số label 1:", num_ones)


torch.Size([32, 96000]) torch.Size([32])
Số label 0: 35074
Số label 1: 30220


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65294 entries, 0 to 65293
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   filename   65294 non-null  object
 1   path       65294 non-null  object
 2   fake_type  65294 non-null  object
 3   label      65294 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 2.0+ MB


In [15]:
train_loss_hist, train_acc_hist, val_acc_hist, val_loss_hist = [], [], [], []
num_epoch = 2

In [14]:
def train(
    len_clip=6,
    version=1,
    lr=1e-5,
    weight_decay=1e-4,
    use_balanced_sampler=True,
    batch_size=64,
    num_workers=4,
):
    if len_clip != 6:
        raise ValueError("This training pipeline is configured for 6s clips only.")
    if version < 1 or version > len(hyper_params):
        raise ValueError(f"Version must be in [1, {len(hyper_params)}]")

    hyper_param = hyper_params[version - 1].copy()
    selected_name = variant_names[version - 1]

    df = load_manifest(TRAIN_CSV)

    df_train, df_val = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    print(f"Variant: {selected_name} (version={version})")
    print(f"Train data: {df_train.shape}")
    print(f"Validation data: {df_val.shape}")

    train_loss_hist.clear()
    train_acc_hist.clear()
    val_acc_hist.clear()
    val_loss_hist.clear()

    train_data = SonicDataset(df_train, duration_seconds=6.0)
    val_data = SonicDataset(df_val, duration_seconds=6.0)

    if batch_size < len(DEVICE_IDS):
        raise ValueError(f"batch_size must be at least {len(DEVICE_IDS)} for DataParallel")

    loader_kwargs = {
        "batch_size": batch_size,  # Global batch size: 32 samples per T4 at the default of 64.
        "num_workers": num_workers,
        "pin_memory": True,
        "persistent_workers": num_workers > 0,
    }
    train_loader = DataLoader(train_data, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_data, shuffle=False, **loader_kwargs)

    model = SpecTTTra(
        **hyper_param,
        expected_samples=train_data.expected_samples,
        num_classes=1 #test probability model
    ).to(device)

    # Splits each batch along dimension 0 across cuda:0 and cuda:1, then gathers outputs on cuda:0.
    model = nn.DataParallel(model, device_ids=DEVICE_IDS, output_device=DEVICE_IDS[0])

    #loss = nn.CrossEntropyLoss()
    loss = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    for epoch in range(num_epoch):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_sample = 0
        train_pred_count = torch.zeros(2, dtype=torch.long)

        for batch in train_loader:
            X = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True).float().view(-1, 1)

            optimizer.zero_grad()
    
            logits = model(X)
            cur_loss = loss(logits, y)
            cur_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += cur_loss.item() * X.size(0)
            total_sample += X.size(0)

            pred = (torch.sigmoid(logits) >= 0.5).long().squeeze(1)
            y_int = y.long().squeeze(1)
            total_correct += (pred == y_int).sum().item()
            train_pred_count += torch.bincount(pred.detach().cpu(), minlength=2)

        epoch_loss = total_loss / total_sample
        epoch_acc = total_correct / total_sample

        # Evaluate the validation split (df_val) after each training epoch.
        model.eval()
        val_total_loss = 0.0
        val_sample = 0
        val_correct = 0
        val_pred_count = torch.zeros(2, dtype=torch.long)
        val_cm = torch.zeros((2, 2), dtype=torch.long)
        with torch.no_grad():
            for batch in val_loader:
                X = batch["x"].to(device, non_blocking=True)
                y = batch["y"].to(device, non_blocking=True).float().view(-1, 1)

                logits = model(X)
                val_batch_loss = loss(logits, y)
                pred = (torch.sigmoid(logits) >= 0.5).long().squeeze(1)
                val_total_loss += val_batch_loss.item() * X.size(0)
                val_sample += X.size(0)
                y_int = y.long().squeeze(1)
                val_correct += (y_int == pred).sum().item()
                val_pred_count += torch.bincount(pred.detach().cpu(), minlength=2)
                val_cm += torch.bincount((y_int.detach().cpu() * 2 + pred.detach().cpu()), minlength=4).reshape(2, 2)

        val_loss = val_total_loss / val_sample
        val_acc = val_correct / val_sample
        recall_fake = val_cm[0, 0].item() / max(1, val_cm[0].sum().item())
        recall_real = val_cm[1, 1].item() / max(1, val_cm[1].sum().item())
        balanced_acc = (recall_fake + recall_real) / 2.0

        train_loss_hist.append(epoch_loss)
        train_acc_hist.append(epoch_acc)
        val_acc_hist.append(val_acc)
        val_loss_hist.append(val_loss)

        print(f"Epoch [{epoch+1}/{num_epoch}] "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Train Acc: {epoch_acc:.4f} | "
            f"Validation Loss: {val_loss:.4f} | "
            f"Validation Acc: {val_acc:.4f} | "
            f"Bal Acc: {balanced_acc:.4f} | "
            f"Recall[Fake/Real]: [{recall_fake:.4f}, {recall_real:.4f}] | "
            f"Train Pred[0/1]: {train_pred_count.tolist()} | "
            f"Validation Pred[0/1]: {val_pred_count.tolist()} | "
            f"Validation CM [[TN,FP],[FN,TP]]: {val_cm.tolist()}")

    return model



In [ ]:
model = train()

# DataParallel prefixes parameter names with `module.`. Save the underlying model so this
# checkpoint can be loaded by either the one-GPU or the two-GPU version of the model.
model_to_save = model.module if isinstance(model, nn.DataParallel) else model
state_dict_cpu = {
    name: tensor.detach().cpu().contiguous()
    for name, tensor in model_to_save.state_dict().items()
}
model_path = OUTPUT_DIR / "model.safetensors"
save_file(state_dict_cpu, str(model_path))
print(f"Saved {model_path}")

Variant: alpha (version=1)
Train data: (52235, 4)
Validation data: (13059, 4)


# Eval

In [ ]:
def evaluate(
    model,
    csv_path=TEST_CSV,
    duration_seconds=6.0,
    batch_size=32,
    num_workers=2,
    threshold=0.5,
    device=None,
):
    # =========================
    # Load CSV
    # =========================
    df = load_manifest(Path(csv_path))

    required_cols = ["filename", "path", "fake_type", "label"]
    df = df[required_cols].copy()

    # Resolve path
    df["path"] = df["path"].astype(str).apply(lambda p: str(Path(p)))

    # Chỉ giữ file tồn tại
    exists_mask = df["path"].apply(lambda p: Path(p).exists())
    if exists_mask.sum() < len(df):
        print(f"[WARN] Missing files: {len(df) - exists_mask.sum()} / {len(df)}")

    df = df[exists_mask].reset_index(drop=True)

    if len(df) == 0:
        raise ValueError("No valid audio files found.")

    df["label"] = df["label"].astype(int)

    # =========================
    # DataLoader
    # =========================
    test_ds = SonicDataset(df, duration_seconds=duration_seconds)

    test_dl = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )

    # =========================
    # Device + eval mode
    # =========================
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model.eval()

    # =========================
    # Predict loop
    # =========================
    all_rows = []

    total = 0
    correct = 0

    tn = fp = fn = tp = 0

    with torch.no_grad():
        for batch in test_dl:
            x = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True).long()

            logits = model(x)

            # Case 1: binary output [B, 1]
            if logits.ndim == 2 and logits.shape[1] == 1:
                prob_real = torch.sigmoid(logits).squeeze(1)
                pred = (prob_real >= threshold).long()

            # Case 2: two-class output [B, 2]
            elif logits.ndim == 2 and logits.shape[1] == 2:
                probs = torch.softmax(logits, dim=1)
                prob_real = probs[:, 1]
                pred = torch.argmax(probs, dim=1)

            else:
                raise ValueError(f"Unexpected model output shape: {logits.shape}")

            total += y.size(0)
            correct += (pred == y).sum().item()

            y_cpu = y.cpu()
            pred_cpu = pred.cpu()
            prob_cpu = prob_real.cpu()

            for i in range(len(y_cpu)):
                label_i = int(y_cpu[i])
                pred_i = int(pred_cpu[i])

                if label_i == 0 and pred_i == 0:
                    tn += 1
                elif label_i == 0 and pred_i == 1:
                    fp += 1
                elif label_i == 1 and pred_i == 0:
                    fn += 1
                elif label_i == 1 and pred_i == 1:
                    tp += 1

                all_rows.append({
                    "filename": batch["filename"][i],
                    "path": batch["path"][i],
                    "fake_type": batch.get("fake_type", [""] * len(y_cpu))[i] if isinstance(batch, dict) else "",
                    "label": label_i,
                    "pred": pred_i,
                    "prob_real": float(prob_cpu[i]),
                })

    # =========================
    # Metrics
    # =========================
    accuracy = correct / total if total > 0 else 0

    recall_fake = tn / (tn + fp) if (tn + fp) > 0 else 0
    recall_real = tp / (tp + fn) if (tp + fn) > 0 else 0

    precision_fake = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_real = tp / (tp + fp) if (tp + fp) > 0 else 0

    f1_fake = (
        2 * precision_fake * recall_fake / (precision_fake + recall_fake)
        if (precision_fake + recall_fake) > 0 else 0
    )

    f1_real = (
        2 * precision_real * recall_real / (precision_real + recall_real)
        if (precision_real + recall_real) > 0 else 0
    )

    balanced_accuracy = (recall_fake + recall_real) / 2

    metrics = {
        "num_samples": total,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,

        "fake_precision": precision_fake,
        "fake_recall": recall_fake,
        "fake_f1": f1_fake,

        "real_precision": precision_real,
        "real_recall": recall_real,
        "real_f1": f1_real,

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "confusion_matrix": [[tn, fp], [fn, tp]],
    }

    pred_df = pd.DataFrame(all_rows)

    print("========== Evaluation ==========")
    print(f"Samples: {total}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
    print()
    print(f"Fake  - Precision: {precision_fake:.4f} | Recall: {recall_fake:.4f} | F1: {f1_fake:.4f}")
    print(f"Real  - Precision: {precision_real:.4f} | Recall: {recall_real:.4f} | F1: {f1_real:.4f}")
    print()
    print("Confusion Matrix [[TN, FP], [FN, TP]]")
    print([[tn, fp], [fn, tp]])

    return metrics, pred_df

In [ ]:
metrics, pred_df = evaluate(
    model,
    csv_path=TEST_CSV,
    batch_size=32,
    duration_seconds=6.0
)

pred_df.head()